In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
sklearn.set_config(display='text')

댓글을 군집화해 분석해 볼 것이다. 댓글 분석은 왜 필요할까? 분석해서 어디에 활용할 수 있을까? 다음과 같은 상황을 생각해 보자.

수백 수천 개 댓글을 다 읽어야 한다면?  
댓글 속에 제품에 대한 관심을 빈도수로 추출해야 한다면?  
쇼핑몰에서 제품 관련 이벤트를 진행할 때 고객이 어떤 제품을 선호하는지 알고 싶다면?  
고객 DB와 연계해 이벤트나 마케닝 세그먼트로 활용한다면?  
향후 마케팅 전략을 세울 때 활용한다면?

데이터는 IT 교육 사이트인 인프런의 새해 다짐 이벤트 댓글을 사용한다. 정답 레이블이 없는 데이터를 분류하고 시각화하는 방법에 초점을 맞출 것이다. 분석하려는 데이터는 정답 레이블이 없으므로 비지도 학습 군집화를 실습한다.

# 데이터 불러와서 전처리 하기

## 데이터 불러오기

In [2]:
df = pd.read_csv('./data/inflearn-event.csv')
df.shape

(2449, 1)

In [3]:
df.head()

,text
0,2020년 목표: 스프링 열심히 공부하서 직장에서 사랑받고 싶어요!!\r\n관심강의...
1,"2020년 목표: C++ 열심히 공부해서, 학교에서 꼭 A 맞기..!! \r\n관심..."
2,2020년 목표 : 리액트 공부하기
3,40대 프로그래밍 시작! : 우리를 위한 프로그래밍 : 파이썬 중급
4,2020년 목표 : 돌머리 비전공자가 멋진 전공자 되기!


In [4]:
df.tail()

,text
2444,"작년 한해도 일이 바쁘다, 야근해서 힘들다는 핑계로 ***님의 JPA 강의를 또 스..."
2445,저는 졸업을 1년 남기고 있는 컴퓨터공학과 학생입니다. 졸업 프로젝트로 웹/앱 개발...
2446,"에프터 이펙트를 써본 적은 있는데, 매번 기초만 배우다 말았어요. 이걸 할 줄 안다..."
2447,저번에 인프런에서 페이스북 마케팅 강의를 듣고 많은 도움을 받았습니다. 마케팅 업무...
2448,인프런 0호 팀원이에요!\r\n그동안 서비스 개발 때문에 js 를 많이 했었는데 앞...


## 중복된 글 제거하기

온라인으로 수집한 데이터는 다양한 이유로 중복 생성 될 수 있다. 웹사이트에서 전송 버튼을 여러 번 누르거나, 새로 고침을 하거나, 네트워크나 UX 관련된 오류 문제가 발생되기도 한다. 중복 데이터가 있으면 빈도 분석이 제대로 되지 않기 때문에 중복 데이터를 제거한다.

중복되는 데이터가 있는지 확인한다.

In [5]:
print(len(df.text), len(set(df.text)))

2449 2410


판다스의 drop_duplicates() 메소드로 데이터의 중복을 제거할 수 있다.  
keep 속성의 기본값은 `first`이고 중복되는 데이터에 첫 번째 데이터를 제외하고 모두 제거하고 `last`는 마지막 데이터를 제외하고 모두 제거한다. 만약에 중복되는 데이터를 모두 제거하고 싶다면 `False`를 사용하면 된다.

In [6]:
print(df.shape)
df = df.drop_duplicates(['text'], keep='last')
print(df.shape)

(2449, 1)
(2410, 1)


## 소문자로 변환하기

전처리할 때는 원본을 따로 보존하는 것을 추천한다. 원본과 전처리 결과를 비교해 볼 수 있고 망쳤을 때 복원할 수 있기 때문이다.

In [7]:
df['origin_text'] = df.text
df.head()

,text,origin_text
0,2020년 목표: 스프링 열심히 공부하서 직장에서 사랑받고 싶어요!!\r\n관심강의...,2020년 목표: 스프링 열심히 공부하서 직장에서 사랑받고 싶어요!!\r\n관심강의...
1,"2020년 목표: C++ 열심히 공부해서, 학교에서 꼭 A 맞기..!! \r\n관심...","2020년 목표: C++ 열심히 공부해서, 학교에서 꼭 A 맞기..!! \r\n관심..."
3,40대 프로그래밍 시작! : 우리를 위한 프로그래밍 : 파이썬 중급,40대 프로그래밍 시작! : 우리를 위한 프로그래밍 : 파이썬 중급
4,2020년 목표 : 돌머리 비전공자가 멋진 전공자 되기!,2020년 목표 : 돌머리 비전공자가 멋진 전공자 되기!
5,2020 년목표: 비전공자(경영)가 전공자(it) 되기!!!,2020 년목표: 비전공자(경영)가 전공자(it) 되기!!!


파이썬은 같은 단어라도 대문자냐 소문자냐에 따라 다른 글자로 인식하므로 lower() 메소드로 모두 소문자로 변경한다.

In [8]:
df['text'] = df.text.str.lower()
df.head()

,text,origin_text
0,2020년 목표: 스프링 열심히 공부하서 직장에서 사랑받고 싶어요!!\r\n관심강의...,2020년 목표: 스프링 열심히 공부하서 직장에서 사랑받고 싶어요!!\r\n관심강의...
1,"2020년 목표: c++ 열심히 공부해서, 학교에서 꼭 a 맞기..!! \r\n관심...","2020년 목표: C++ 열심히 공부해서, 학교에서 꼭 A 맞기..!! \r\n관심..."
3,40대 프로그래밍 시작! : 우리를 위한 프로그래밍 : 파이썬 중급,40대 프로그래밍 시작! : 우리를 위한 프로그래밍 : 파이썬 중급
4,2020년 목표 : 돌머리 비전공자가 멋진 전공자 되기!,2020년 목표 : 돌머리 비전공자가 멋진 전공자 되기!
5,2020 년목표: 비전공자(경영)가 전공자(it) 되기!!!,2020 년목표: 비전공자(경영)가 전공자(it) 되기!!!


'파이썬', 'python'과 같이 의미는 같으나 표기가 다르게 되어 있는 단어도 하나로 통일하기 위해서 replace() 메소드로 치환한다.

In [9]:
df['text'] = df.text.str.replace('python', '파이썬').str.replace('react', '리액트').str.replace('pandas', '판다스').str.replace('javascript', '자바스크립트'
    ).str.replace('java', '자바').str.replace('mongodb', '몽고DB').str.replace('django', '장고')
df.head()

,text,origin_text
0,2020년 목표: 스프링 열심히 공부하서 직장에서 사랑받고 싶어요!!\r\n관심강의...,2020년 목표: 스프링 열심히 공부하서 직장에서 사랑받고 싶어요!!\r\n관심강의...
1,"2020년 목표: c++ 열심히 공부해서, 학교에서 꼭 a 맞기..!! \r\n관심...","2020년 목표: C++ 열심히 공부해서, 학교에서 꼭 A 맞기..!! \r\n관심..."
3,40대 프로그래밍 시작! : 우리를 위한 프로그래밍 : 파이썬 중급,40대 프로그래밍 시작! : 우리를 위한 프로그래밍 : 파이썬 중급
4,2020년 목표 : 돌머리 비전공자가 멋진 전공자 되기!,2020년 목표 : 돌머리 비전공자가 멋진 전공자 되기!
5,2020 년목표: 비전공자(경영)가 전공자(it) 되기!!!,2020 년목표: 비전공자(경영)가 전공자(it) 되기!!!


'관심강의'라는 텍스트가 있다. '관심 있는 강의', '관심있는 강의', '#관심강의' 등으로 다양하게 표현돼있어서 아래와 같이 replace() 메소드로 치환하려 했었으나 다양하게 표현된 텍스트가 너무 많아서 cav 파일 자체를 수정했다. ㅠㅠ

In [10]:
df['text'] = df.text.str.replace('관심 있는 강의', '관심강의').str.replace('관심있는 강의', '관심강의').str.replace('#관심강의', '관심강의')

'관심강의'를 기준으로 split() 메소드로 분할하면 학생이 관심있는 강의를 알 수 있다.

In [11]:
# df['course'] = df.text.apply(lambda x: x.split('관심강의')[-1])
df['course'] = df.text.str.split('관심강의').str[-1]
df['course'] = df.course.str.replace(':', '').str.strip()

In [12]:
df[['text', 'course']].head()

,text,course
0,2020년 목표: 스프링 열심히 공부하서 직장에서 사랑받고 싶어요!!\r\n관심강의...,스프링 웹 mvc
1,"2020년 목표: c++ 열심히 공부해서, 학교에서 꼭 a 맞기..!! \r\n관심...",***c++
3,40대 프로그래밍 시작! : 우리를 위한 프로그래밍 : 파이썬 중급,40대 프로그래밍 시작! 우리를 위한 프로그래밍 파이썬 중급
4,2020년 목표 : 돌머리 비전공자가 멋진 전공자 되기!,2020년 목표 돌머리 비전공자가 멋진 전공자 되기!
5,2020 년목표: 비전공자(경영)가 전공자(it) 되기!!!,2020 년목표 비전공자(경영)가 전공자(it) 되기!!!


search_keyword에 course 열에 저장된 관심강의에서 검색할 과목 이름을 저장하고 반복문을 실행해서 관심강의에 검색할 과목 이름을 contains() 메소드로 포함하고 있을 경우 검색어를 열 이름으로 하는 열에 True를 그렇치 않으면 False를 넣어준다.

In [13]:
search_keyword = ['파이썬', '크롤링', '넘파이', '판다스', '시각화', '데이터분석', '몽고DB', '머신러닝', '딥러닝', '자연어', 'llm', 'rag', 'langgraph', 'neo4j']
for keyword in search_keyword:
    df[keyword] = df.text.str.contains(keyword)

In [14]:
df.head()

,text,origin_text,course,파이썬,크롤링,넘파이,판다스,시각화,데이터분석,몽고DB,머신러닝,딥러닝,자연어,llm,rag,langgraph,neo4j
0,2020년 목표: 스프링 열심히 공부하서 직장에서 사랑받고 싶어요!!\r\n관심강의...,2020년 목표: 스프링 열심히 공부하서 직장에서 사랑받고 싶어요!!\r\n관심강의...,스프링 웹 mvc,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,"2020년 목표: c++ 열심히 공부해서, 학교에서 꼭 a 맞기..!! \r\n관심...","2020년 목표: C++ 열심히 공부해서, 학교에서 꼭 A 맞기..!! \r\n관심...",***c++,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,40대 프로그래밍 시작! : 우리를 위한 프로그래밍 : 파이썬 중급,40대 프로그래밍 시작! : 우리를 위한 프로그래밍 : 파이썬 중급,40대 프로그래밍 시작! 우리를 위한 프로그래밍 파이썬 중급,True,False,False,False,False,False,False,False,False,False,False,False,False,False
4,2020년 목표 : 돌머리 비전공자가 멋진 전공자 되기!,2020년 목표 : 돌머리 비전공자가 멋진 전공자 되기!,2020년 목표 돌머리 비전공자가 멋진 전공자 되기!,False,False,False,False,False,False,False,False,False,False,False,False,False,False
5,2020 년목표: 비전공자(경영)가 전공자(it) 되기!!!,2020 년목표: 비전공자(경영)가 전공자(it) 되기!!!,2020 년목표 비전공자(경영)가 전공자(it) 되기!!!,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [19]:
df[df.머신러닝 & df.딥러닝].head()

,text,origin_text,course,파이썬,크롤링,넘파이,판다스,시각화,데이터분석,몽고DB,머신러닝,딥러닝,자연어,llm,rag,langgraph,neo4j
45,sw 개발자입니다. 자기개발 및 업무 환경 개선을 위해 공부하고 싶습니다.\r\n관...,SW 개발자입니다. 자기개발 및 업무 환경 개선을 위해 공부하고 싶습니다.\r\n관...,"모두를 위한 딥러닝 - deep reinforcement learning, 그 외 ...",True,False,False,False,False,False,False,True,True,False,False,False,False,False
59,올해 졸업하기 전에 머신러닝을 마스터하고 싶습니다.\r\n관심강의: 모두를 위한 딥...,올해 졸업하기 전에 머신러닝을 마스터하고 싶습니다.\r\n관심강의: 모두를 위한 딥...,"모두를 위한 딥러닝 - deep reinforcement learning, 그 외 ...",True,False,False,False,False,False,False,True,True,False,False,False,False,False
236,금융인을 꿈꾸는 인문대생인데 인프런을 통해 그 꿈에 날개를 달 수 있으면 좋겠습니다...,금융인을 꿈꾸는 인문대생인데 인프런을 통해 그 꿈에 날개를 달 수 있으면 좋겠습니다...,금융인을 꿈꾸는 인문대생인데 인프런을 통해 그 꿈에 날개를 달 수 있으면 좋겠습니다...,False,False,False,False,False,False,False,True,True,False,False,False,False,False
262,인문게 졸업생인데 공학 계열로 진로를 변경하려 합니다. 인프런에서 '모두를 위한 딥...,인문게 졸업생인데 공학 계열로 진로를 변경하려 합니다. 인프런에서 '모두를 위한 딥...,r로 하는 웹 크롤링 - 입문편,False,True,False,False,False,False,False,True,True,False,False,False,False,False
279,올해 새롭게 딥러닝에 대한 공부를 시작하고자 합니다!\r\n관심강의 : 밑바닥부터 ...,올해 새롭게 딥러닝에 대한 공부를 시작하고자 합니다!\r\n관심강의 : 밑바닥부터 ...,"밑바닥부터 시작하는 머신러닝 입문, 공공 데이터 분석",False,False,False,False,False,False,False,True,True,False,False,False,False,False


이번에는 정규표현식을 사용해 '파이썬', '판다스', '데이터분석'라는 텍스트가 들어간 데이터를 찾아보자. contains() 메소드를 사용하면 정규표현식을 사용해 원하는 키워드가 들어간 데이터를 찾을 수 있다.

In [69]:
df_python = df[df.text.str.contains(r'.*(파이썬|판다스|데이터분석).*')]
df_python.shape

(435, 17)

In [71]:
df[df.text.str.contains(r'.*(머신러닝|딥러닝).*')].shape

(184, 17)

댓글에 특정 키워드가 포함되어있나 검색해서 있으면 True(1), 없으면 False(0)를 넣은 파생 변수들의 True의 빈도수를 sum() 메소드로 계산한다.

In [75]:
df[search_keyword].sum().sort_values(ascending=False)

파이썬          428
머신러닝         155
딥러닝           73
크롤링           59
데이터분석         36
시각화           30
판다스            6
자연어            4
몽고DB           2
넘파이            0
llm            0
rag            0
langgraph      0
neo4j          0
dtype: int64

In [79]:
text = df.loc[df.데이터분석, 'text']
for t in text:
    print(t)
    print('-' * 100)

웹개발, 데이터분석 하고 싶어서 수업수강 하게 되었어요
관심강의 :  파이썬 입문 수강 중이니까 수강 후에 결정할래용
----------------------------------------------------------------------------------------------------
r 데이터분석의 전문가가 되고싶어용
관심강의 : r로 하는 텍스트마이닝 (top keyword부터 감성분석까지)
----------------------------------------------------------------------------------------------------
퍼포먼스 마케터를 꿈꾸는 학생입니다. 데이터분석 특히 시각화 공부와 더불어서 디지털마케팅전반에 대해 공부하고 싶네요.
관심강의 : 파이썬데이터시각화 분석 실전 프로젝트 , 그로스해킹 - 데이터와 실험을 통해 성장하는 서비스를 만드는 방법
----------------------------------------------------------------------------------------------------
데이터분석가로 변신을 꿈꾸는 직장인입니다. 올해는 r과 파이썬을 마스터하고 싶습니다. 기초 코딩부터 시작해서 머신러닝과 딥러닝까지 도전해보고 싶습니다! 인프런의 강좌들이 많은 도움이 될 것 같습니다! 열공하겠습니다! ^^
관심강의 : r, 파이썬  관련 강의
----------------------------------------------------------------------------------------------------
파이썬 데이터분석 마스터
관심강의:공공데이터로 파이썬 데이터 분석 시작하기
----------------------------------------------------------------------------------------------------
이제 막 군대 전역하는 24살 대학생입니다. 대학 전공은 바이오메디컬이지만 데이터분

In [80]:
text = df.loc[df.판다스, 'text']
for t in text:
    print(t)
    print('-' * 100)

2020년에는 데이터분석 관련한 실력을 쌓고싶습니다!
관심강의 : 파이썬, 판다스, 데이터분석, 머신러닝
----------------------------------------------------------------------------------------------------
취미로 안드로이드 개발하는 사람입니다. 자바로 작성하다 보니, kotlin이 안드로이드 기반언어로 바뀌어서 새로 배워보려고 합니다.
관심강의 : kotlin, 판다스, 파이썬, c언어
----------------------------------------------------------------------------------------------------
판다스 라입러리에 관심이 많아서 배워보려 합니다 관심강의 : *** 판다스
----------------------------------------------------------------------------------------------------
2020년!! 올 해는 빅데이터 분석 전문가 되기!!
관심강의 : 파이썬, 판다스
----------------------------------------------------------------------------------------------------
2020년 목표  파이썬 을  활용해  데이터 분석 작업을 진행하고 싶습니다.  관심강의:파이썬 판다스 관련 강좌 입니다
----------------------------------------------------------------------------------------------------
저는 백세시대에 조금 더 오랫동안 it일을 하기위해서 it전략기획에서 데이터분석가로 커리어 전환을 준비하고 있습니다. 인프런의 수많은 좋은 강의중 파이썬 기반의 데이터분석을 공부하기 위하여 인프런을 활용하고 있습니다. 대한민국 최고의 데이터분석가 되는 그날까지 인프런과 함께 하겟습니다.
관심강의 : 파이썬 판다스 데

# 벡터화 하기